In [1]:
import os

In [2]:
%pwd

'/Users/yashwanthreddygotike/Documents/Projects/Text-Summarization-NLP-Project/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/yashwanthreddygotike/Documents/Projects/Text-Summarization-NLP-Project'

Now Update config.yaml - for data ingestion

#entity for data ingestion - to store all the variables related to data ingestion


In [5]:
#entity for data ingestion - to store all the variables related to data ingestion
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
#update config manager in src/config/configuration.py to read the config.yaml file 
# and create an instance of DataIngestionConfig class 
# update constanst first.py file to add the path of config.yaml file and params.yaml file

from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH, 
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

components

In [8]:
#update components/data_ingestion.py file to read the config.yaml file and create an instance of DataIngestionConfig class and print the values of the variables in the config.yaml file
import os
import urllib.request as request
import zipfile
from textSummarizer.utils.common import get_size
from textSummarizer.logging import logger

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
         # ✅ FIX: create directory first
        os.makedirs(self.config.root_dir, exist_ok=True)
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL, 
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extract the zip file to the data directory
        Function returns none
        """
        unzip_path = self.config.unzip_dir
        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

In [10]:
#create pipeline for data ingestion and test the pipeline
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    logger.info(f"Data Ingestion Config: {data_ingestion_config}")
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-04-07 17:55:03,676: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-07 17:55:03,678: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-07 17:55:03,679: INFO: common: created directory at: artifacts]
[2026-04-07 17:55:03,679: INFO: 2417050603: Data Ingestion Config: DataIngestionConfig(root_dir='artifacts/data_ingestion', source_URL='https://github.com/ycrgotike/datasets/raw/refs/heads/main/textsummarizer-data.zip', local_data_file='artifacts/data_ingestion/data.zip', unzip_dir='artifacts/data_ingestion')]
[2026-04-07 17:55:03,680: INFO: 2093803592: File already exists of size: ~ 8081 KB]
